In [1]:
import nltk
from nltk.stem import PorterStemmer,WordNetLemmatizer
import re

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\cvhir\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
text = "OMG!!! I can't open https://abc.com :)"

text = text.lower()
text = re.sub(r"https?://\S+|www\.\S+","",text)

text = text.replace("can't","cannot")
text = text.replace("'re","are")
text = text.replace("won't","will not")
text = text.replace("'ve","have")

text = text.replace(":)","happy")
text = text.replace(":(","sad")

tokens = re.findall(r"\b\w+\b",text)

stemmer = PorterStemmer()
stems = [stemmer.stem(word) for word in tokens]

lemmatizer = WordNetLemmatizer()
lemma = [lemmatizer.lemmatize(word,pos ="v") for word in tokens]

print("cleaned : ",text)
print("tokens : ",tokens)
print("stems : ",stems)
print("lemma : " , lemma)

cleaned :  omg!!! i cannot open  happy
tokens :  ['omg', 'i', 'cannot', 'open', 'happy']
stems :  ['omg', 'i', 'cannot', 'open', 'happi']
lemma :  ['omg', 'i', 'cannot', 'open', 'happy']


In [3]:
import random
import math
from collections import Counter

In [11]:
sentences =[
    "i love nlp",
    "i love python",
    "i like nlp",
    "you love python"
]

tokens_list = [
    re.findall(r"\b\w+\b",s.lower())
    for s in sentences
]

bigrams = Counter()
context = Counter()

for tokens in tokens_list:
    tokens = ["<s>"]+tokens+["</s>"]

    for i in range(len(tokens)-1):
        bigrams[(tokens[i],tokens[i+1])] += 1
        context[(tokens[i])] += 1

vocab = set(word for tokens in tokens_list for word in tokens)
vocab.add("</s>")
V=len(vocab)

def bigram_prob(w1,w2):
    return ((bigrams[(w1,w2)]+1)/(context[w1] + V))

def sentence_prob(sentence):
    words = re.findall(r"\b\w+\b",sentence.lower())

    words = ["<s>"]+words+["</s>"]


    prob =1.0

    for i in range(len(words) - 1):
        prob *= bigram_prob(words[i],words[i+1])

    return prob

def perplexity(sentence):

    words = re.findall(r"\b\w+\b",sentence.lower())
    
    words = ["<s>"]+words+["</s>"]

    log_prob = 0.0

    N = len(words)-1

    for i in range(N):
        p = bigram_prob(words[i],words[i+1])
        log_prob += math.log2(p)

    return math.exp(-log_prob/N)

print("vocab : ",vocab)
print("bigrams : ",bigrams)
print("P(love | i):", bigram_prob("i", "love"))

print("sentence probability : ",sentence_prob("i love nlp"))
print("perplexity : ",perplexity("i love nlp"))
    



vocab :  {'like', 'you', 'love', 'i', '</s>', 'python', 'nlp'}
bigrams :  Counter({('<s>', 'i'): 3, ('i', 'love'): 2, ('nlp', '</s>'): 2, ('love', 'python'): 2, ('python', '</s>'): 2, ('love', 'nlp'): 1, ('i', 'like'): 1, ('like', 'nlp'): 1, ('<s>', 'you'): 1, ('you', 'love'): 1})
P(love | i): 0.3
sentence probability :  0.007272727272727273
perplexity :  5.90513287848402


In [ ]:

import nltk

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

sentence = "The little boy plays happily"

words = nltk.word_tokenize(sentence)
tags = nltk.pos_tag(words)

print("Words:", words)
print("POS Tags:")

for word, tag in tags:
    print(word, ":", tag)


In [ ]:

import nltk

grammar = nltk.CFG.fromstring("""
S -> NP VP
NP -> PRP
NP -> DT NN
NP -> NP PP
VP -> V NP
VP -> VP PP
PP -> P NP
PRP -> 'I'
DT -> 'the'
NN -> 'man' | 'telescope'
V -> 'saw'
P -> 'with'
""")

parser = nltk.ChartParser(grammar)

sentence = "I saw the man with the telescope"
words = sentence.split()

trees = list(parser.parse(words))

print("Number of parse trees:", len(trees))

for tree in trees:
    tree.pretty_print()


In [ ]:

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
The food was delicious.
The service was friendly.
I liked the excellent camera.
"""

doc = nlp(text)

print("Dependencies:")

for token in doc:
    print(token.text, token.dep_, token.head.text)

print("\nAspect-Opinion Pairs:")

pairs = []

for token in doc:
    # Example: food -> delicious
    if token.dep_ == "nsubj" and token.head.pos_ == "ADJ":
        pairs.append((token.text, token.head.text))

    # Example: camera -> excellent
    if token.dep_ == "amod":
        pairs.append((token.head.text, token.text))

print(pairs)


In [ ]:

import nltk
from nltk.wsd import lesk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")

sentence = "He deposited money in the bank"
words = nltk.word_tokenize(sentence)

sense = lesk(words, "bank", "n")

print("Sentence:", sentence)

if sense:
    print("Selected sense:", sense.name())
    print("Definition:", sense.definition())
else:
    print("No sense found")


In [ ]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")

word1 = "dog"
word2 = "dog"

syn1 = wn.synsets(word1, pos=wn.NOUN)[0]
syn2 = wn.synsets(word2, pos=wn.NOUN)[0]

print("Synset 1:", syn1.name())
print("Synset 2:", syn2.name())
print("Are they identical?", syn1 == syn2)
print("Similarity:", syn1.wup_similarity(syn2))

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "I love machine learning",
    "I love deep learning",
    "I love cricket"
]

query = "machine learning"

# 1. Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

# 2. Convert query into a TF-IDF vector
query_vector = vectorizer.transform([query])

# 3. Calculate cosine similarity
scores = cosine_similarity(query_vector, doc_vectors)[0]

# 4. Retrieve top-k documents
k = 2
top_indices = scores.argsort()[::-1][:k]

print("Query:", query)
print("\nSimilarity scores:")

for i, score in enumerate(scores):
    print(documents[i], ":", round(score, 3))

print("\nTop", k, "documents:")

for i in top_indices:
    print(documents[i], ":", round(scores[i], 3))


In [ ]:

import re
import math
from collections import Counter

text = "i love machine learning and i love deep learning"

words = re.findall(r"\b\w+\b", text.lower())

window = 2
pair_counts = Counter()
word_counts = Counter()

for i, word in enumerate(words):
    word_counts[word] += 1

    for j in range(max(0, i - window),
                   min(len(words), i + window + 1)):

        if i != j:
            pair_counts[(word, words[j])] += 1

total_pairs = sum(pair_counts.values())
total_words = sum(word_counts.values())

print("PPMI values:")

for (word, context), count in pair_counts.items():
    p_wc = count / total_pairs
    p_w = word_counts[word] / total_words
    p_c = word_counts[context] / total_words

    pmi = math.log2(p_wc / (p_w * p_c))
    ppmi = max(0, pmi)

    print(word, context, ":", round(ppmi, 3))


In [ ]:

from gensim.models import Word2Vec
import re

sentences = [
    "i love machine learning",
    "i love deep learning",
    "machine learning is useful",
    "deep learning is interesting"
]

tokenized = [
    re.findall(r"\b\w+\b", s.lower())
    for s in sentences
]

model = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=2,
    min_count=1,
    sg=1,
    seed=42,
    workers=1,
    epochs=100
)

print("Vector for machine:")
print(model.wv["machine"])

print("\nSimilar words to learning:")
print(model.wv.most_similar("learning"))


In [ ]:

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
Alice met Bob.
She greeted him.
Alice thanked him.
"""

doc = nlp(text)

# 1. Hard-coded gender information
female_names = ["alice", "sara", "mary"]
male_names = ["bob", "ravi", "john"]

# 2. Find mentions: names, noun chunks and pronouns
mentions = {}

for ent in doc.ents:
    if ent.label_ == "PERSON":
        mentions[(ent.start, ent.end)] = ent

for chunk in doc.noun_chunks:
    mentions[(chunk.start, chunk.end)] = chunk

for token in doc:
    if token.pos_ == "PRON":
        mentions[(token.i, token.i + 1)] = token.doc[
            token.i:token.i + 1
        ]

mentions = sorted(
    mentions.values(), key=lambda m: m.start
)

# 3. Get gender
def get_gender(mention):
    word = mention.text.lower().replace(".", "").split()[-1]

    if word in female_names or word in ["she", "her", "hers"]:
        return "female"

    if word in male_names or word in ["he", "him", "his"]:
        return "male"

    # Title clues
    text_lower = mention.text.lower()

    if text_lower.startswith(("mr ", "mr.", "mr. ")):
        return "male"

    if text_lower.startswith(("mrs ", "mrs.", "mrs. ")):
        return "female"

    return "unknown"

# 4. Get number
def get_number(mention):
    if mention.root.pos_ == "PRON":
        if mention.root.lower_ in ["they", "them", "we", "us"]:
            return "plural"
        return "singular"

    if mention.root.pos_ == "PROPN":
        return "singular"

    if mention.root.tag_ in ["NNS", "NNPS"]:
        return "plural"

    return "singular"

# 5. Score an earlier mention
def score_pair(previous, current):
    score = 0

    g1 = get_gender(previous)
    g2 = get_gender(current)

    if g1 != "unknown" and g2 != "unknown":
        if g1 == g2:
            score += 2
        else:
            score -= 2

    if get_number(previous) == get_number(current):
        score += 2
    else:
        score -= 2

    if previous.root.lemma_ == current.root.lemma_:
        score += 2

    distance = current.sent.start - previous.sent.start
    score -= max(0, distance // 5)

    return score

# 6. Build coreference chains
chains = []

for i, mention in enumerate(mentions):
    best = None
    best_score = 1

    for previous in mentions[:i]:
        score = score_pair(previous, mention)

        if score > best_score:
            best_score = score
            best = previous

    if best is not None:
        # Add to the chain containing the antecedent
        found = False

        for chain in chains:
            if best in chain:
                chain.append(mention)
                found = True
                break

        if not found:
            chains.append([best, mention])

# 7. Display mentions and chains
print("Mentions:")
for mention in mentions:
    print(mention.text, "->", get_gender(mention),
          ",", get_number(mention))

print("\nCoreference Chains:")
for chain in chains:
    print([m.text for m in chain])


In [ ]:

pip install transformers torch

from transformers import pipeline

# Create fill-mask pipeline
fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "I drink a cup of [MASK]."

results = fill_mask(text)

for result in results:
    print("Word:", result["token_str"])
    print("Probability:", result["score"])
    print()


In [ ]:

from transformers import pipeline

# Load sentiment analysis pipeline
sentiment = pipeline("sentiment-analysis")

texts = [
    "I love this movie",
    "This movie is terrible"
]

for text in texts:
    result = sentiment(text)[0]

    print("Text:", text)
    print("Sentiment:", result["label"])
    print("Score:", result["score"])
    print()


In [ ]:

import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# 1. Dataset: 9 samples
texts = [
    "I love this movie",
    "This is a wonderful film",
    "The acting was excellent",
    "I really enjoyed it",
    "Amazing and beautiful story",
    "I hate this movie",
    "This film is terrible",
    "The acting was bad",
    "I really disliked it"
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0]

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels
)

# 3. Load tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. Tokenize the text
train_encodings = tokenizer(
    X_train, truncation=True, padding=True
)
test_encodings = tokenizer(
    X_test, truncation=True, padding=True
)

# 5. Create datasets
train_data = Dataset.from_dict({
    **train_encodings,
    "labels": y_train
})

test_data = Dataset.from_dict({
    **test_encodings,
    "labels": y_test
})

# 6. Load BERT for classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 7. Evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0)
    }

# 8. Training settings
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    report_to="none",
    seed=42
)

# 9. Train the model
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()

# 10. Evaluate
print(trainer.evaluate())

# 11. Predict new text
new_text = ["The movie was fantastic"]

inputs = tokenizer(
    new_text, return_tensors="pt",
    truncation=True, padding=True
)

import torch

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(
    outputs.logits, dim=1
).tolist()

for text, pred in zip(new_text, predictions):
    label = "Positive" if pred == 1 else "Negative"
    print(text, "->", label)
